In [15]:
import numpy as np
from tensorflow.keras.models import load_model,Model
from sklearn.preprocessing import normalize

In [16]:
#Dataset yükle

X = np.load("../data/X.npy")
y_act = np.load("../data/y_act.npy")
y_user = np.load("../data/y_user.npy")

print("X shape:",X.shape)
print("y_user shape:")



X shape: (3521, 300, 90)
y_user shape:


In [17]:
unique_users= np.unique(y_user)
print("Unique users:",unique_users)
print("Total users:",len(unique_users))

#6 user --> Train
#1 user --> known test
#1 user --> unknown test


Unique users: [0 1 2 3 6 7 8 9]
Total users: 8


In [18]:
train_users=[0,1,2,3,6,7]
known_user = 8
unknown_user= 9

print("Train users:",train_users)
print("Known test user:",known_user)
print("Unknown test user:",unknown_user)


Train users: [0, 1, 2, 3, 6, 7]
Known test user: 8
Unknown test user: 9


In [35]:
#Train mask 
train_mask =np.isin(y_user,train_users)

#Known test mask (görülmüş ama train'e dahil değil)
known_mask = (y_user == known_user)

#Unknown test mask(model hiç görmeyecek)
unknown_mask =(y_user ==unknown_user)

#Datasetleri ayıralaım
#X_train --> Triplet eğitimi için
X_train = X[train_mask]
y_train = y_user[train_mask]

#X_known--> Model tanımalı
X_known = X[known_mask]
y_known = y_user[known_mask]

#X_unknown-->Model unknown deneli
X_unknown= X[unknown_mask]
y_unknown= y_user[unknown_mask]


print("Train shape:", X_train.shape)
print("Known test shape:", X_known.shape)
print("Unknown test shape:", X_unknown.shape)

#Open-set'in temeli atıldı

Train shape: (2881, 300, 90)
Known test shape: (480, 300, 90)
Unknown test shape: (160, 300, 90)


In [20]:
#normalizasyon yapalım

mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train-mean)/std
X_known = (X_known-mean)/std
X_unknown=(X_unknown-mean)/std

print("Normalizasyon tamamlandı.")

Normalizasyon tamamlandı.


In [21]:
#Activity modeli yükleyelim

activity_model = load_model("../models/activity_model.h5")
activity_model.summary()
activity_model.build((None,300,90))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 300, 64)        │        28,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 300, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 150, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 150, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 75, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 75, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         2,064 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 188,882 (737.82 KB)

 Trainable params: 187,984 (734.31 KB)

 Non-trainable params: 896 (3.50 KB)

 Optimizer params: 2 (12.00 B)

In [22]:
#Softmaz'tan önceki katmanı al 
#Embedding modeli çıkaralım

embedding_model=Model(
    inputs=activity_model.layers[0].input,
    outputs=activity_model.layers[-2].output
)
embedding_model.summary()

Model: "functional_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 300, 90)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 300, 64)        │        28,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 300, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 150, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 150, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 75, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 75, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,816 (729.75 KB)

 Trainable params: 185,920 (726.25 KB)

 Non-trainable params: 896 (3.50 KB)

In [23]:
#Embeddingleri üretelim

train_embeddings = embedding_model.predict(X_train,verbose=0)
known_embeddings = embedding_model.predict(X_known,verbose=0)
unknown_embeddings = embedding_model.predict(X_unknown,verbose=0)

print("Train:",train_embeddings.shape)
print("Known:",known_embeddings.shape)
print("Unknow:",unknown_embeddings.shape)



Train: (2881, 128)
Known: (480, 128)
Unknow: (160, 128)


In [24]:
#Embeddingleri normalize edelim

train_embeddings= normalize(train_embeddings,axis=1)
known_embeddings= normalize(known_embeddings,axis=1)
unknown_embeddings= normalize(unknown_embeddings,axis=1)

print("Embedding normalization tamam.")

Embedding normalization tamam.


In [37]:
gallery = {}
gallery_labels={}

for user in train_users:
    user_mask = (y_train == user)
    user_embeddings = train_embeddings[user_mask]
    
    #Ortalama embedding
    mean_embedding = np.mean(user_embeddings,axis=0)
    mean_embedding =mean_embedding / np.linalg.norm(mean_embedding)

    gallery[user] = mean_embedding
    
print("Galley oluşturuldu.")
print("Gallery users:",list(gallery.keys()))   

Galley oluşturuldu.
Gallery users: [0, 1, 2, 3, 6, 7]


In [ ]:
#Known accuracy 

def cosine_similarity(a,b):
    return np.dot(a,b)

correct = 0

for emb in known_embeddings:
    sims= {user:cosine_similarity(emb,gallery[user]) for user in gallery}
    
    pred_user =max(sims, key =sims.get)
    
    if sims[pred_user] > 0.65:
        correct += 1
        
print("Known accuracy:",correct/len(known_embeddings))

# User 8 gallery'de yok.
# Ancak embedding uzayı kullanıcıya göre optimize edilmediği için
# model bazen 8'i gallery'deki bir kullanıcıya benzetebiliyor.

Known accuracy: 0.5354166666666667
